# Implementación Base de un LLM (GPT-2)


Se reúne en un solo notebook la implementación base completa de un LLM autorregresivo basado en la arquitectura **Transformer *decoder-only*** de lenguaje de gran tamaño tipo GPT-2. Desarrollada a lo largo de los cuadernos **`ch02.ipynb`, `ch03.ipynb`, `ch04.ipynb`, `ch05.ipynb` y `ch06.ipynb`** del repositorio *Build a Large Language Model From Scratch* (Sebastian Raschka).

Posteriormente, se modifica la implementación, sin llegar a expandirla o a eliminar código original (se deja mencionado) para adecuarla a nuestro proyecto.


## 1. Explicación

### 1.1. Qué se usa realmente de cada capítulo


En los notebooks originales del libro se construyen primero versiones didácticas simplificadas o de juguete y se intercalan descargas de datasets y ejecuciones de prueba para verificar cada paso. En este archivo se copian únicamente las clases, funciones y configuraciones definitivas necesarias para construir el LLM desde cero.

| Archivo origen | Qué usamos nosotros (copiado de los notebooks) | Qué se descarta del notebook original (y por qué) |
| :--- | :--- | :--- |
| **`ch02.ipynb`** | • Inicialización del tokenizador BPE `tiktoken.get_encoding("gpt2")` (§2.5).<br>• Clase **`GPTDatasetV1`** y función **`create_dataloader_v1`** (§2.6). | • Tokenizadores manuales `SimpleTokenizerV1` (§2.3) y `SimpleTokenizerV2` (§2.4), sustituidos por BPE.<br>• Descarga de `the-verdict.txt`, pruebas de codificación de frases sueltas y ejemplos de `nn.Embedding` (§2.2, §2.5, §2.7–§2.8). |
| **`ch03.ipynb`** | • Clase definitiva **`MultiHeadAttention`** con división de pesos (*weight splits*), máscara causal y *dropout* (§3.6.2). | • Versiones previas `softmax_naive` (§3.3), `SelfAttention_v1`, `SelfAttention_v2` (§3.4), `CausalAttention` (§3.5.3) y `MultiHeadAttentionWrapper` (§3.6.1).<br>• Tensores de prueba de 6 tokens para imprimir dimensiones. |
| **`ch04.ipynb`** | • Diccionario **`GPT_CONFIG_124M`** (§4.1).<br>• Clases **`LayerNorm`** (§4.2), **`GELU`** y **`FeedForward`** (§4.3).<br>• Clases **`TransformerBlock`** (§4.5) y **`GPTModel`** (§4.6).<br>• Función **`generate_text_simple`** (§4.7). | • Clases esqueleto `DummyGPTModel`, `DummyTransformerBlock` y `DummyLayerNorm` (§4.1).<br>• Red de juguete `ExampleDeepNeuralNetwork` (§4.4) y ejecuciones de prueba sobre tensores aleatorios o modelos sin entrenar. |
| **`ch05.ipynb`** | • Funciones **`text_to_token_ids`** y **`token_ids_to_text`** (§5.1.1).<br>• Funciones **`calc_loss_batch`** y **`calc_loss_loader`** (§5.1.3).<br>• Funciones **`train_model_simple`**, **`evaluate_model`**, **`generate_and_print_sample`** y **`plot_losses`** (§5.2).<br>• Funciones **`softmax_with_temperature`** y **`generate`** (§5.3).<br>• Diccionario **`model_configs`** y funciones **`assign`** y **`load_weights_into_gpt`** (§5.5). | • Cálculos manuales de `torch.log` (§5.1.2) y `print_sampled_tokens` (§5.3.1).<br>• Entrenamiento de ejemplo sobre `the-verdict.txt`, pruebas de guardado `.pth` (§5.4) y ejecuciones de descarga/generación. |
| **`ch06.ipynb`** | • Funciones **`create_balanced_dataset`** y **`random_split`** (§6.2).<br>• Clase **`SpamDataset`** (§6.3).<br>• Función **`prepare_model_for_classification`** (§6.5).<br>• Función **`calc_accuracy_loader`** (§6.6).<br>• Funciones **`train_classifier_simple`**, **`plot_values`** y **`classify_review`** (§6.7–§6.8). | • Descarga del dataset SMS Spam Collection, creación de `.csv` de ejemplo, pruebas *zero-shot* (§6.4) y ejecuciones de entrenamiento/inferencia sobre frases de prueba. |


### 1.2. Modificaciones respecto a la implementación original


Todo el código base de este cuaderno procede de las celdas de `ch02.ipynb` a `ch06.ipynb` (omitiendo los `from previous_chapters import ...` y todas las celdas de ejemplo/ejecución sobre datasets de prueba), con las siguientes modificaciones y añadidos propios:

1. **Unificación de `calc_loss_batch` y soporte para máscara `-100` (`IGNORE_INDEX`)**: En `ch06.ipynb` se redefinían `calc_loss_batch`, `calc_loss_loader` y `evaluate_model` con el mismo nombre que en `ch05.ipynb`. Se mantienen una sola vez en la Sección 5, añadiendo en `calc_loss_batch` la condición `if target_batch.ndim == 2:` junto con `ignore_index=-100` (para ignorar en la pérdida los tokens de la pregunta y del *padding*) y aceptando en `calc_loss_loader` lotes tanto de 2 elementos `(x, y)` como de 3 elementos `(x, y, level)`.
2. **Encapsulación en `prepare_model_for_classification` (Sección 6.3)**: Las sentencias sueltas de congelación de parámetros y reemplazo de `model.out_head` de `ch06.ipynb` se encapsulan dentro de la función `prepare_model_for_classification(model, base_config, num_classes=2)`.
3. **Extracción de estados ocultos y pesos de atención (`return_hidden`, `return_attn`, `head` en Secciones 3.1, 4.4 y 4.5)**:
   - En `MultiHeadAttention.forward` y `TransformerBlock.forward` se añade el parámetro `return_attn=False`. Cuando es `True`, devuelve los pesos de atención `(b, n_heads, T, T)` inmediatamente después del `softmax` y antes del `dropout`. Si es `False`, no guarda tensores adicionales y el resultado es idéntico.
   - En `GPTModel` se exponen los métodos `embed(in_idx)` y `head(h)` (para *logit lens*: `self.out_head(self.final_norm(h))`), y en `GPTModel.forward(in_idx, n_loops=None, return_hidden=False, return_attn=False)` se recorre `self.trf_blocks` mediante un bucle `for` en lugar de invocar `nn.Sequential` de golpe. Los nombres de los parámetros (`state_dict`) no cambian, pero permite devolver `(logits, extras)` con `extras["hidden"]` ($L+1$ estados `[h0, h1, ..., hL]`) y `extras["attn"]` ($L$ tensores de atención).
4. **Tokenizador por caracteres (`CharTokenizer`), `MathDataset` y configuración reducida (`BASE_CONFIG`) (Secciones 2.3 y 4.1)**:
   - En la Sección 2.3 se añaden la clase `CharTokenizer` de vocabulario fijo (~100 símbolos: `<pad>=0`, `<eos>=1`, `<unk>=2` y caracteres del español y signos matemáticos) y la clase `MathDataset` con su `collate_math_batch` (que formatea `"Pregunta: ...\nRespuesta: ..."` y asigna objetivo `-100` a la pregunta y al relleno).
   - En la Sección 4.1 se añade `BASE_CONFIG` (`emb_dim=256`, `n_heads=8`, `context_length=512`, `drop_rate=0.0`, `qkv_bias=False`), dimensionada para entrenar modelos ligeros desde cero en GPUs de 4 GB.

6. **Almacenamiento de `self.cfg` y contabilidad de parámetros y FLOPs (`count_params`, `block_flops_per_token`, `gpt_flops_per_token`, Sección 4.7)**:
   - `GPTModel` almacena una copia de su configuración en `self.cfg` al inicializarse para que cualquier utilidad de análisis, guardado o conversión pueda consultar sus hiperparámetros arquitectónicos directamente desde la instancia.
   - Se añaden en la Sección 4.7 las funciones `count_params(model, include_embeddings=True, trainable_only=True)`, `block_flops_per_token(cfg, seq_len=None)` ($24 d^2 + 4 T d$) y `gpt_flops_per_token(cfg, seq_len=None)` para cuantificar el tamaño y coste computacional por token del modelo base.


## 2. Procesamiento de Texto, Tokenización y DataLoader



Definimos tanto el tokenizador **Byte-Pair Encoding (BPE)** de GPT-2 (`tiktoken`) con `GPTDatasetV1` (`ch02.ipynb`), como el tokenizador por caracteres de vocabulario fijo **`CharTokenizer`** y el dataset **`MathDataset`** para problemas matemáticos.



### 2.1. Tokenizador Byte-Pair Encoding (BPE) con `tiktoken`

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

### 2.2. Muestreo de datos con ventana deslizante: `GPTDatasetV1` y `create_dataloader_v1`

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

### 2.3. Tokenizador por caracteres (`CharTokenizer`) y Dataset de problemas matemáticos (`MathDataset`)

In [ ]:
import json

IGNORE_INDEX = -100


class CharTokenizer:
    """Tokenizador por caracteres con vocabulario fijo.
    IDs: <pad> = 0, <eos> = 1, <unk> = 2, seguidos de espacio, salto de línea, dígitos,
    letras inglesas, vocales acentuadas/diéresis/ñ y signos de puntuación y aritméticos."""

    SPECIAL_TOKENS = ["<pad>", "<eos>", "<unk>"]
    CHARS = (
        " \n"
        + "0123456789"
        + "abcdefghijklmnopqrstuvwxyz"
        + "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        + "áéíóúüñÁÉÍÓÚÜÑ"
        + ".,;:¿?¡!()-\"'+×="
    )

    def __init__(self):
        self.pad_id = 0
        self.eos_id = 1
        self.unk_id = 2
        self.itos = list(self.SPECIAL_TOKENS) + list(self.CHARS)
        self.stoi = {ch: idx for idx, ch in enumerate(self.itos)}
        self.vocab_size = len(self.itos)

    def encode(self, text, allowed_special=None):
        return [self.stoi.get(ch, self.unk_id) for ch in text]

    def decode(self, token_ids):
        out = []
        for tid in token_ids:
            tid = int(tid)
            if tid in (self.pad_id, self.eos_id):
                continue
            if tid == self.unk_id or tid < 0 or tid >= self.vocab_size:
                out.append("")
            else:
                out.append(self.itos[tid])
        return "".join(out)


char_tokenizer = CharTokenizer()

In [ ]:
def format_math_prompt(question):
    """Construye el prefijo de entrada hasta 'Respuesta: ' incluido."""
    return f"Pregunta: {question}\nRespuesta: "


class MathDataset(Dataset):
    """Dataset para ficheros JSONL de problemas matemáticos.
    La pérdida se calcula únicamente sobre los caracteres de la respuesta y el token <eos>
    (todo lo anterior lleva objetivo IGNORE_INDEX = -100)."""

    def __init__(self, jsonl_path_or_records, tokenizer=None, context_length=512):
        self.tokenizer = char_tokenizer if tokenizer is None else tokenizer
        self.context_length = context_length
        if isinstance(jsonl_path_or_records, (str, bytes)) or hasattr(jsonl_path_or_records, "__fspath__"):
            with open(jsonl_path_or_records, "r", encoding="utf-8") as f:
                self.records = [json.loads(line) for line in f if line.strip()]
        else:
            self.records = list(jsonl_path_or_records)

        self.samples = []
        for rec in self.records:
            prefix = format_math_prompt(rec["question"])
            answer_str = str(rec["answer"])
            prefix_ids = self.tokenizer.encode(prefix)
            ans_ids = self.tokenizer.encode(answer_str) + [self.tokenizer.eos_id]
            full_ids = (prefix_ids + ans_ids)[: self.context_length + 1]

            x = torch.tensor(full_ids[:-1], dtype=torch.long)
            y = torch.tensor(full_ids[1:], dtype=torch.long)
            # Los primeros len(prefix_ids) - 1 objetivos corresponden a caracteres dentro de la pregunta
            n_ignore = min(len(prefix_ids) - 1, len(y))
            y[:n_ignore] = IGNORE_INDEX
            level = int(rec.get("level", 1))
            self.samples.append((x, y, level))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_math_batch(batch, pad_id=0):
    """Relleno dinámico por la derecha a la longitud máxima del batch."""
    xs, ys, levels = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    x_batch = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    y_batch = torch.full((len(ys), max_len), IGNORE_INDEX, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        x_batch[i, : x.size(0)] = x
        y_batch[i, : y.size(0)] = y
    return x_batch, y_batch, torch.tensor(levels, dtype=torch.long)


def create_math_dataloader(jsonl_path_or_records, tokenizer=None, batch_size=64,
                           context_length=512, shuffle=True, drop_last=False, num_workers=0):
    tok = char_tokenizer if tokenizer is None else tokenizer
    dataset = MathDataset(jsonl_path_or_records, tokenizer=tok, context_length=context_length)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        collate_fn=lambda b: collate_math_batch(b, pad_id=tok.pad_id),
    )

## 3. Mecanismo de Atención Causal Multi-Cabeza


La clase `MultiHeadAttention` proyecta las entradas mediante tres únicas matrices `W_query`, `W_key` y `W_value` de dimensión `(d_in, d_out)` y divide el último eje en `(num_heads, head_dim)` mediante `.view()` y `.transpose(1, 2)`, calculando la atención causal escalada de todas las cabezas en una sola operación matricial. Si `return_attn=True`, devuelve además los pesos de atención `(b, num_heads, num_tokens, num_tokens)` después del `softmax` y antes del `dropout`.



### 3.1. Implementación eficiente con división de pesos: `MultiHeadAttention`

In [ ]:
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x, return_attn=False):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_dropped = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_dropped @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        if return_attn:
            return context_vec, attn_weights
        return context_vec

## 4. Arquitectura Completa del Modelo GPT desde Cero

Se ensamblan todas las piezas arquitectónicas del modelo GPT-2 (`GPT_CONFIG_124M` y la configuración reducida `BASE_CONFIG`): `LayerNorm`, `GELU` y `FeedForward`, `TransformerBlock`, `GPTModel` (con soporte para `return_hidden`, `return_attn` y `head`) y la función de generación *greedy* `generate_text_simple`.



### 4.1. Configuraciones base: `GPT_CONFIG_124M` y `BASE_CONFIG`

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

# Configuración reducida por defecto (~0.8M params/bloque, apta para entrenar desde cero en GPUs de 4 GB)
BASE_CONFIG = {
    "vocab_size": char_tokenizer.vocab_size,
    "context_length": 512,
    "emb_dim": 256,
    "n_heads": 8,
    "n_layers": 4,
    "drop_rate": 0.0,
    "qkv_bias": False,
}

### 4.2. Normalización por capas: `LayerNorm`

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

### 4.3. Activación `GELU` y red `FeedForward`

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device, dtype=x.dtype)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

### 4.4. Conexión de atención y capas lineales en un bloque Transformer: `TransformerBlock`

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x, return_attn=False):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        if return_attn:
            x, attn_weights = self.att(x, return_attn=True)
        else:
            x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        if return_attn:
            return x, attn_weights
        return x

### 4.5. Codificación del modelo GPT completo: `GPTModel`

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def embed(self, in_idx):
        """Calcula el estado inicial h0 = Dropout(TokEmb(in_idx) + PosEmb(0..T-1))."""
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        return self.drop_emb(tok_embeds + pos_embeds)

    def head(self, h):
        """Aplica final_norm + out_head sobre cualquier estado oculto (para logit lens)."""
        return self.out_head(self.final_norm(h))

    def forward(self, in_idx, n_loops=None, return_hidden=False, return_attn=False):
        x = self.embed(in_idx)
        if not return_hidden and not return_attn:
            for block in self.trf_blocks:
                x = block(x)
            return self.head(x)

        extras = {}
        if return_hidden:
            extras["hidden"] = [x]
        if return_attn:
            extras["attn"] = []

        for block in self.trf_blocks:
            if return_attn:
                x, attn = block(x, return_attn=True)
                extras["attn"].append(attn)
            else:
                x = block(x)
            if return_hidden:
                extras["hidden"].append(x)

        return self.head(x), extras

### 4.6. Generación de texto (*greedy decoding*): `generate_text_simple`

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

### 4.7. Contabilidad de parámetros y FLOPs por token: `count_params`, `block_flops_per_token` y `gpt_flops_per_token`

Definimos las funciones para contar el número de parámetros entrenables de cualquier instancia de `GPTModel` y calcular analíticamente los FLOPs del paso hacia adelante (*forward*) por token tanto de un bloque `TransformerBlock` individual ($24 d^2 + 4 T d$) como del modelo `GPTModel` completo de $L$ capas.

In [ ]:
def count_params(model, include_embeddings=True, trainable_only=True):
    """Cuenta el número de parámetros del modelo (por defecto, los entrenables incluyendo embeddings)."""
    params = (p for p in model.parameters() if (p.requires_grad or not trainable_only))
    total = sum(p.numel() for p in params)
    if not include_embeddings:
        total -= model.tok_emb.weight.numel() + model.pos_emb.weight.numel()
    return total


def block_flops_per_token(cfg, seq_len=None):
    """FLOPs aproximados (2 * MACs) en el pase forward por token para un único `TransformerBlock`:
    - Proyecciones Q, K, V y salida: 4 * (2 * d^2) = 8 * d^2
    - Atención QK^T y AV:            2 * (2 * T * d) = 4 * T * d
    - Red FeedForward (d -> 4d -> d): 2 * (2 * 4 * d^2) = 16 * d^2
    Total por bloque = 24 * d^2 + 4 * T * d."""
    d = int(cfg["emb_dim"])
    T = int(cfg.get("context_length", 512) if seq_len is None else seq_len)
    return 24 * (d ** 2) + 4 * T * d


def gpt_flops_per_token(cfg, seq_len=None):
    """FLOPs totales de forward por token para un `GPTModel` de L capas (incluyendo la cabeza `out_head`)."""
    L = int(cfg["n_layers"])
    d = int(cfg["emb_dim"])
    V = int(cfg["vocab_size"])
    return L * block_flops_per_token(cfg, seq_len=seq_len) + 2 * d * V


## 5. Preentrenamiento en Datos No Etiquetados, Evaluación y Pesos de OpenAI

Se implementan las funciones de conversión entre texto y tokens, el cálculo de pérdida por entropía cruzada, el bucle de preentrenamiento `train_model_simple`, la función de decodificación estocástica `generate` con temperatura y *top-k*, y la transferencia de pesos preentrenados de OpenAI.


### 5.1. Evaluación de modelos de texto generativos



#### 5.1.1. Utilidades de conversión: `text_to_token_ids` y `token_ids_to_text`

In [ ]:
def text_to_token_ids(text, tokenizer):
    try:
        encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    except TypeError:
        encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded, dtype=torch.long).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

#### 5.1.2. Cálculo de las pérdidas de entrenamiento y validación: `calc_loss_batch` y `calc_loss_loader`

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    if target_batch.ndim == 2:  # Preentrenamiento / problemas matemáticos (ignora objetivos -100)
        logits = model(input_batch)
        loss = torch.nn.functional.cross_entropy(
            logits.flatten(0, 1), target_batch.flatten(), ignore_index=IGNORE_INDEX
        )
    else:  # Cap. 6: clasificación (solo el último token)
        logits = model(input_batch)[:, -1, :]
        loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, batch in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = batch[0], batch[1]
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

### 5.2. Entrenamiento de un LLM


#### 5.2.1. Bucle principal: `train_model_simple`, `evaluate_model` y `generate_and_print_sample`

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        
        for batch in train_loader:
            input_batch, target_batch = batch[0], batch[1]
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

#### 5.2.2. Visualización de pérdidas: `plot_losses`

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

### 5.3. Estrategias de decodificación para controlar la aleatoriedad: Temperatura y Top-$k$

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # New (not in book): numerical stability tip to get equivalent results on mps device
            # subtract rowwise max before softmax
            logits = logits - logits.max(dim=-1, keepdim=True).values
            
            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if eos_id is not None and (idx_next == eos_id).all():  # Stop generating early if end-of-sequence token is encountered
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

### 5.4. Carga de pesos preentrenados de OpenAI

In [ ]:
try:
    from gpt_download import download_and_load_gpt2
except ImportError:
    download_and_load_gpt2 = None


In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

## 6. Fine-Tuning para Clasificación de Texto



Se prepara el dataset con *padding* (`SpamDataset`), se reemplaza `model.out_head` por una capa `nn.Linear(emb_dim, 2)` descongelando el último bloque Transformer y la `LayerNorm` final, y se entrena optimizando la entropía cruzada sobre los *logits* del **último token** `model(input_batch)[:, -1, :]`.


### 6.1. Balanceo y partición del dataset: `create_balanced_dataset` y `random_split`

In [ ]:
import pandas as pd


def create_balanced_dataset(df):
    
    # Count the instances of "spam"
    num_spam = df[df["Label"] == "spam"].shape[0]
    
    # Randomly sample "ham" instances to match the number of "spam" instances
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    
    # Combine ham "subset" with "spam"
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df

In [ ]:
def random_split(df, train_frac, validation_frac):
    # Shuffle the entire DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Calculate split indices
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

### 6.2. Dataset con *padding*: `SpamDataset`

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # Pre-tokenize texts
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncate sequences if they are longer than max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

### 6.3. Adaptación de la cabeza de clasificación: `prepare_model_for_classification`

In [ ]:
def prepare_model_for_classification(model, base_config, num_classes=2):
    for param in model.parameters():
        param.requires_grad = False

    torch.manual_seed(123)

    model.out_head = torch.nn.Linear(in_features=base_config["emb_dim"], out_features=num_classes)

    for param in model.trf_blocks[-1].parameters():
        param.requires_grad = True

    for param in model.final_norm.parameters():
        param.requires_grad = True

    return model

### 6.4. Cálculo de exactitud: `calc_accuracy_loader`

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

### 6.5. Ajuste fino del modelo con datos supervisados: `train_classifier_simple` y `plot_values`

In [ ]:
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    # Initialize lists to track losses and examples seen
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            examples_seen += input_batch.shape[0] # New: track examples instead of tokens
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Calculate accuracy after each epoch
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [ ]:
def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label="Validation {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"Validation {label}")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # Create a second x-axis for examples seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(examples_seen, train_values, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

### 6.6. Inferencia del clasificador: `classify_review`

In [ ]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # Prepare inputs to the model
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    # Truncate sequences if they too long
    input_ids = input_ids[:min(max_length, supported_context_length)]
    assert max_length is not None, (
        "max_length must be specified. If you want to use the full model context, "
        "pass max_length=model.pos_emb.weight.shape[0]."
    )
    assert max_length <= supported_context_length, (
        f"max_length ({max_length}) exceeds model's supported context length ({supported_context_length})."
    )
    
    # Pad sequences to the longest sequence
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # add batch dimension

    # Model inference
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Logits of the last output token
    predicted_label = torch.argmax(logits, dim=-1).item()

    # Return the classified result
    return "spam" if predicted_label == 1 else "not spam"